<div style="border-left:5px solid #67CBEE;padding:10px 16px;background:#F7F9FB;color:#19222D;margin:12px 0"><b>AMD Radeon · YOLO26 · Day 01</b><br><span style="color:#536273">算法 → 数据 → 训练 → 评测 → 证据。每一个结论都应由实际运行结果支撑。</span><br><span style="color:#536273">Algorithm → Data → Training → Evaluation → Evidence. Every conclusion must be backed by real execution results.</span></div>

# 05 / Lab E · 受控 A/B 训练优化实验

**建议时长：30–40 分钟（完整训练可能更久）**　｜　学习路径：30–40 minutes (full training may take longer)

> 请按顺序运行；运行结果必须来自自己的 GPU 环境，不可把示例文字当作实测数据。

---

# 05 / Lab E · Controlled A/B Training Experiment

**Suggested time: 控制变量 → 重新训练 → 同口径比较** | Learning path: controlled variable → retrain → same-protocol comparison

> Run the cells in order; every result must come from your own GPU environment — never present the sample text as measured data.

## 实验目标
基于 Lab C 的**真实 baseline**，只修改训练图像尺寸（512 → 416）；保持模型初始化权重、数据集、Epoch、Batch、随机种子等条件不变，用统一的验证输入尺寸 640 测量两组模型。不要因为速度更快就直接宣布算法优化成功。

> **公平性说明：** 降低 imgsz 会改变输入信息量和计算量。固定相同 Epoch 不保证相同训练时长 / SGD 更新质量；本实验验证一个明确的课程假设，不提供统计显著性证明。训练时间还可能受缓存及云端 GPU 并发负载影响。

---

## Goal
Starting from the **real baseline produced by Lab C**, change exactly one thing — the training image size (512 → 416). Keep initial weights, dataset, epochs, batch, random seed and everything else identical, then measure both models with the same validation input size of 640. Never declare an optimization successful just because it runs faster.

> **Fairness note:** lowering imgsz changes both the information content and the compute of each step. Equal epochs do not guarantee equal training time / SGD update quality; this experiment tests one clearly stated classroom hypothesis and provides no statistical significance. Training time is also affected by caches and concurrent load on shared cloud GPUs.

In [ ]:
from pathlib import Path
import os, sys, json
candidates = [Path.cwd(), Path.cwd().parent, Path('/workspace/YOLO26_ROCm_Day1_Labs')]
package = next((p for p in candidates if (p/'course/lab_utils.py').exists()), None)
if package is None:
    raise FileNotFoundError('请先将 ZIP 完整解压，并从项目目录或 notebooks/ 目录启动 Jupyter。')
sys.path.insert(0, str(package))
from course.lab_utils import (root, ensure_dirs, load_config, save_config, write_json,
    write_csv, system_info, require_rocm, sha256)
home = ensure_dirs()
cfg = load_config()
print('Project:', package, '\nPersistent output:', home)

In [ ]:
from course.lab_utils import get_metrics_summary
import torch,time
from ultralytics import YOLO
device=require_rocm()
assert cfg.get('baseline_train_settings') and Path(cfg['baseline_weights']).is_file(),'请先完整运行 Lab C baseline。'
base_record=json.loads((home/'reports/lab_c_baseline.json').read_text(encoding='utf-8'))
args=cfg['baseline_train_settings']
print('Baseline train imgsz:',args['imgsz'],'epochs:',args['epochs'])
print('Baseline model sha256:',sha256(Path(cfg['baseline_weights'])))
model_cache=home/'artifacts/yolo26n.pt'
assert model_cache.is_file(), '预训练初始权重缓存不存在；重新运行 00 进行固定。'
assert sha256(model_cache)==cfg.get('model_sha256'), '初始权重 SHA256 改变，A/B 结果不可比。'
assert args['imgsz']==512,'此 Notebook 的实验方案假定 baseline 训练尺寸为 512，请先审核与修改方案'

In [ ]:
# Variant：只修改 imgsz；额外不同的仅是 run output name/project
RUN_VARIANT=True
variant_args=dict(args)
variant_args.update(imgsz=416,project=str(home/'runs/lab_e'),name='imgsz416',exist_ok=True)
changed={key:(args[key],variant_args[key]) for key in args if args[key]!=variant_args[key]}
print('Changed training arguments:',changed)
assert set(changed)=={'imgsz','project','name'}
if RUN_VARIANT:
    torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
    t0=time.perf_counter()
    init_model=str(home/'artifacts/yolo26n.pt') if (home/'artifacts/yolo26n.pt').is_file() else cfg['model']
    variant=YOLO(init_model)  # 注意：不能用 baseline best.pt 作为 B 的初始权重！
    variant.train(**variant_args)
    torch.cuda.synchronize()
    run_dir=Path(variant.trainer.save_dir)
    variant_best=run_dir/'weights/best.pt'
    assert variant_best.is_file()
    variant_record={'run_dir':str(run_dir),'weights':str(variant_best),
       'sha256':sha256(variant_best),'elapsed_seconds':round(time.perf_counter()-t0,2),
       'peak_allocated_GiB':round(torch.cuda.max_memory_allocated()/2**30,3),
       'args':variant_args,'system':system_info()}
    write_json(home/'reports/lab_e_variant.json',variant_record)
    print('Variant training finished:',variant_record)
else:
    print('RUN_VARIANT=False: without real variant artifacts NO A/B conclusion can be made.')

In [ ]:
# 相同验证集、验证输入和运行条件，分别评测真实权重。
variant_record=json.loads((home/'reports/lab_e_variant.json').read_text(encoding='utf-8'))
assert Path(variant_record['weights']).exists()
reports=[]
for label,rec in [('baseline',base_record),('imgsz416',variant_record)]:
    m=YOLO(rec['weights']).val(data=cfg['dataset_yaml'],split='val',
                  imgsz=640,device=device,batch=8,workers=0,plots=False,verbose=False)
    scores=get_metrics_summary(m)
    scores.update({'variant':label,'train_imgsz':rec['args']['imgsz'],
      'training_time_seconds':rec['elapsed_seconds'],'train_peak_allocated_gib':rec['peak_allocated_GiB'],
      'model_sha256':rec['sha256'],'val_imgsz':640,'split':'val'})
    reports.append(scores)
    print(label,scores)
write_csv(home/'reports/lab_e_ab_comparison.csv',reports,
          ['variant','train_imgsz','training_time_seconds','train_peak_allocated_gib',
           'precision','recall','mAP50','mAP50_95','model_sha256','val_imgsz','split'])
write_json(home/'reports/lab_e_ab_comparison.json',reports)

In [ ]:
# 图表使用实测数值，不预置任何提升率
from matplotlib import pyplot as plt
labels=[r['variant'] for r in reports]
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(labels,[r['mAP50_95'] for r in reports])
ax.set_ylabel('Validation mAP50-95');ax.set_ylim(0,1)
ax.set_title('Same validation images and evaluation imgsz=640')
for i,r in enumerate(reports):ax.text(i,r['mAP50_95']+.01,f"{r['mAP50_95']:.3f}",ha='center')
fig.tight_layout()
fig.savefig(home/'assets/lab_e_ab_mAP.png',dpi=150)
plt.show()
print('ΔmAP50-95 B−A:',round(reports[1]['mAP50_95']-reports[0]['mAP50_95'],4))

### 最终 Test：仅在选定模型之后运行一次
只使用 Validation 指标决定选择哪个候选模型。不要在查看 Test 分数后反过来改动超参数，再把同一 Test 分数当作独立最终成绩。

---

### The final Test: run exactly once, after model selection
Use validation metrics alone to pick the winning candidate. Never peek at the test score, go back and tweak hyperparameters, and then present that same test score as an independent final result.

In [ ]:
RUN_FINAL_TEST = False  # 小组完成模型选择并冻结方案后，明确改成 True 执行一次
if RUN_FINAL_TEST:
    target=home/'reports/lab_e_final_test.json'
    assert not target.exists(), '独立 Test 已有成绩。避免重复查看后调参。'
    selection=max(reports,key=lambda row:row['mAP50_95'])  # 仅由 Val 指标选择
    test_weight=base_record['weights'] if selection['variant']=='baseline' else variant_record['weights']
    final_test=YOLO(test_weight).val(data=cfg['dataset_yaml'],split='test',imgsz=640,
                                    device=device,batch=8,workers=0,plots=False,verbose=False)
    result=get_metrics_summary(final_test)
    result.update({'split':'test','selected_by':'validation mAP50-95',
                   'model_variant':selection['variant'],'model_sha256':sha256(Path(test_weight)),
                   'warning':'Tiny classroom split; not a statistically reliable industrial benchmark.'})
    write_json(target,result)
    print('Frozen final test result:',result)
else:
    print('Test is sealed. Current metrics are Validation only.')

### 结论必须回答五个问题
1. 只改了哪一个**算法实验变量**？`project/name` 是日志位置变化，不是算法变量。
2. 在**完全一致的验证集和评测输入尺寸**下，mAP50-95 是增加还是减少？
3. 哪组训练耗时更短？同一云端 GPU 是否有其他用户负载干扰？
4. 显存数字是 PyTorch allocated peak，不是整卡显存占用。
5. 这个结果能否推广到更大模型、不同硬件和工业现场？为什么不能直接断言？

**交付**：`lab_e_variant.json`、`lab_e_ab_comparison.csv`、`lab_e_ab_mAP.png`、文字分析。

**进阶**：用多个随机种子重复实验；在真正独立的业务数据上验证；在第二天再比较 ONNX/OpenVINO、GPU/ROCm 推理速度、显存与准确性。

---

### Your conclusion must answer five questions
1. Which **single experimental variable** changed? `project/name` are log locations, not algorithm variables.
2. Under **identical validation data and evaluation input size**, did mAP50-95 go up or down?
3. Which arm trained faster? Was any other user loading the same cloud GPU?
4. The memory figure is PyTorch's allocated peak, not whole-GPU VRAM usage.
5. Can this result generalize to larger models, other hardware and production sites? Why can't you just assert it?

**Deliverables**: `lab_e_variant.json`, `lab_e_ab_comparison.csv`, `lab_e_ab_mAP.png`, written analysis.

**Going further**: repeat with multiple random seeds; validate on truly independent business data; on day 2 compare ONNX/OpenVINO vs GPU/ROCm inference speed, memory and accuracy.